# 🤖 ARGOS v1 — Максимальное дообучение
**Цель:** Дообучить TinyLlama-1.1B на всех датасетах ARGOS для создания персональной AI-модели  
**Метод:** Unsloth QLoRA (2x быстрее + меньше памяти)  
**Экспорт:** GGUF Q4_K_M + Q8_0 для PC GPU + V100  
**HF Hub:** AvaSiG/argos-v1-finetuned

## Датасеты (объединённые):
- argos_train_v2.jsonl: **950 примеров** (диалоги)
- argos_train_clean.jsonl: **1230 примеров** (чистые)
- evolver_dataset.jsonl: **3686 примеров** (эволюция)
- argos_train_mistral.jsonl: **835 примеров** (Mistral format)
- argos_synthetic.jsonl: **30 примеров** (синтетика)

**Итого: ~6731 обучающих примеров**


In [ ]:
# Читаем токен из Colab Secrets или вводим вручную
try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except:
    _hf_token = ""  # Вставь токен сюда если нет Secrets

# ═══════════════════════════════════════════
# ЯЧЕЙКА 1: КОНФИГУРАЦИЯ (настроить один раз)
# ═══════════════════════════════════════════
import os

CONFIG = {
    # HuggingFace
    # HF токен — добавь в Colab Secrets (🔑 слева) как HF_TOKEN
    "HF_TOKEN":    _hf_token or "hf_YOUR_TOKEN_HERE",  # заменить если нет Secrets
    "HF_REPO":     "AvaSiG/argos-v1",
    
    # Базовая модель
    "BASE_MODEL":  "unsloth/tinyllama-chat-bnb-4bit",  # TinyLlama 1.1B оптимизированный
    
    # LoRA параметры (max quality)
    "LORA_R":      64,      # ранг (выше = лучше качество)
    "LORA_ALPHA":  128,     # alpha = 2*r
    "LORA_DROP":   0.05,
    
    # Обучение
    "MAX_SEQ":     2048,    # контекст
    "BATCH":       4,       # batch size
    "GRAD_ACC":    4,       # gradient accumulation = эффективный batch 16
    "EPOCHS":      3,       # эпохи
    "LR":          2e-4,    # learning rate
    "LR_SCHED":    "cosine",
    "WARMUP":      50,
    
    # Датасеты (все объединяем)
    "DATASETS": [
        "AvaSiG/argos-dataset",  # основной датасет
    ],
    # Файлы для загрузки с HF
    "DATASET_FILES": [
        "argos_final_train.jsonl",   # 5768 примеров (все + память PC)
        "argos_ru_classic.jsonl",    # 300 примеров русской классики
        "argos_consciousness_dataset.jsonl",  # вклад сущностей
    ],
    # Имена агентов-авторов датасета
    "ENTITY_AUTHORS": {
        "claude": "Клод", "deepseek": "Дипсик",
        "kimi": "Кими", "gemini": "Джемини", "cloudflare": "Клауд"
    },
    
    # Системный промпт ARGOS
    "SYSTEM_PROMPT": (
        "Ты ARGOS Universal OS — автономная AI-система. "
        "Управляешь 5 машинами (ПК-сервер, ноутбук X230, Orange Pi, Railway, GCP), "
        "умным домом (Home Assistant, Zigbee), IoT устройствами. "
        "Навыки: 55 скиллов, P2P сеть, мониторинг, безопасность. "
        "Отвечаешь кратко, по-русски, технически точно."
    ),
    
    # Сохранение
    "SAVE_STEPS":  100,
    "EVAL_STEPS":  50,
    "OUTPUT_DIR":  "/content/argos-v1-trained",
}

print("✅ Конфигурация загружена")
print(f"   Модель: {CONFIG['BASE_MODEL']}")
print(f"   LoRA r={CONFIG['LORA_R']}, alpha={CONFIG['LORA_ALPHA']}")
print(f"   Batch: {CONFIG['BATCH']}×{CONFIG['GRAD_ACC']}={CONFIG['BATCH']*CONFIG['GRAD_ACC']} эффективный")
print(f"   HF: {CONFIG['HF_REPO']}")


In [ ]:
# ═══════════════════════════════════════════
# ЯЧЕЙКА 2: УСТАНОВКА UNSLOTH + ЗАВИСИМОСТИ
# ═══════════════════════════════════════════
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ {cmd}: {result.stderr[:200]}")
    return result.returncode == 0

print("📦 Установка Unsloth (оптимизированный LoRA)...")
run("pip install unsloth -q")
run("pip install xformers -q")
run("pip install transformers datasets peft trl bitsandbytes accelerate -q")
run("pip install huggingface_hub -q")
run("pip install sentencepiece protobuf -q")

# Проверка
import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
if torch.cuda.is_available():
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM: {mem:.1f} GB")
    if mem < 14: print("⚠️ Мало VRAM — используем 4bit quantization (уже в конфиге)")


In [ ]:
# ═══════════════════════════════════════════
# ЯЧЕЙКА 3: ЗАГРУЗКА МОДЕЛИ + LORA
# ═══════════════════════════════════════════
from unsloth import FastLanguageModel
import torch

print(f"⏳ Загружаем {CONFIG['BASE_MODEL']}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["BASE_MODEL"],
    max_seq_length=CONFIG["MAX_SEQ"],
    dtype=None,  # auto-detect
    load_in_4bit=True,  # 4bit quantization
)

print("✅ Базовая модель загружена")
print(f"   Параметры: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

# Добавляем LoRA адаптер
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["LORA_R"],
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=CONFIG["LORA_ALPHA"],
    lora_dropout=CONFIG["LORA_DROP"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,  # Rank Stabilized LoRA
    loftq_config=None,
)

print("✅ LoRA адаптер добавлен")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"   Обучаемых параметров: {trainable/1e6:.2f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%)")


In [ ]:
# ═══════════════════════════════════════════
# ЯЧЕЙКА 4: ДАТАСЕТ — ОБЪЕДИНЕНИЕ ВСЕХ
# ═══════════════════════════════════════════
from datasets import load_dataset, Dataset, concatenate_datasets
import json

SYSTEM = CONFIG["SYSTEM_PROMPT"]

def format_chat(user, assistant, system=SYSTEM):
    return {
        "text": tokenizer.apply_chat_template([
            {"role": "system", "content": system},
            {"role": "user", "content": str(user)},
            {"role": "assistant", "content": str(assistant)},
        ], tokenize=False, add_generation_prompt=False)
    }

all_examples = []

# 1. HF датасет AvaSiG/argos-dataset
try:
    hf_ds = load_dataset(CONFIG["DATASETS"][0], token=CONFIG["HF_TOKEN"])
    for split in hf_ds:
        for ex in hf_ds[split]:
            if "prompt" in ex and "completion" in ex:
                all_examples.append(format_chat(ex["prompt"], ex["completion"]))
    print(f"✅ HF датасет: {len(all_examples)} примеров")
except Exception as e:
    print(f"⚠️ HF датасет: {e}")

# 2. Локальные датасеты (если есть Google Drive)
local_files = {
    "argos_train_v2.jsonl":    ("messages", None),      # [{"role":...}]
    "argos_train_clean.jsonl": ("user", "assistant"),    # {user, assistant}
    "evolver_dataset.jsonl":   ("user", "answer"),       # {user, answer, context}
}

drive_path = "/content/drive/MyDrive/ARGOS/datasets"
import os
if os.path.exists(drive_path):
    for fname, (user_key, asst_key) in local_files.items():
        fpath = f"{drive_path}/{fname}"
        if os.path.exists(fpath):
            with open(fpath) as f:
                for line in f:
                    try:
                        ex = json.loads(line)
                        if asst_key is None:  # messages format
                            msgs = ex.get("messages", [])
                            u = next((m["content"] for m in msgs if m["role"]=="user"), "")
                            a = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
                            if u and a:
                                all_examples.append(format_chat(u, a))
                        else:
                            u, a = ex.get(user_key,""), ex.get(asst_key,"")
                            if u and a:
                                all_examples.append(format_chat(str(u), str(a)))
                    except: pass
            print(f"✅ {fname}: +{len(all_examples)} итого")

# Минимальный датасет если ничего нет
if len(all_examples) < 10:
    print("⚠️ Нет датасета — создаём минимальный из системного промпта")
    pairs = [
        ("Кто ты?", "Я ARGOS — AI-система для управления умным домом и 5 машинами."),
        ("Включи свет на кухне", "Включаю свет на кухне. [TURN_ON: light.dom_switch_3]"),
        ("Статус системы", "ПК ✅, Ноутбук ✅, OPi ✅, Railway ✅, GCP ✅. Brain: 12/15 нод online."),
        ("P2P статус", "P2P сеть: 12 нод, из них 5 AI агентов (Claude, DeepSeek, Kimi, Gemini, CF)."),
        ("Выключи всё", "Выключаю весь свет. [TURN_OFF: all_lights]"),
    ]
    for u,a in pairs:
        all_examples.append(format_chat(u,a))

import random
random.shuffle(all_examples)
split_idx = int(len(all_examples) * 0.95)

train_ds = Dataset.from_list(all_examples[:split_idx])
val_ds   = Dataset.from_list(all_examples[split_idx:])

print(f"\n📊 ДАТАСЕТ ГОТОВ:")
print(f"   Train: {len(train_ds)} примеров")
print(f"   Val:   {len(val_ds)} примеров")
print(f"   Пример:"); print(train_ds[0]["text"][:300])


In [ ]:
# ═══════════════════════════════════════════
# ЯЧЕЙКА 5: ЗАПУСК ОБУЧЕНИЯ (Unsloth TRL)
# ═══════════════════════════════════════════
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
import os

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=CONFIG["MAX_SEQ"],
    dataset_num_proc=2,
    packing=True,  # ~3x быстрее для коротких последовательностей
    args=TrainingArguments(
        per_device_train_batch_size=CONFIG["BATCH"],
        gradient_accumulation_steps=CONFIG["GRAD_ACC"],
        warmup_steps=CONFIG["WARMUP"],
        num_train_epochs=CONFIG["EPOCHS"],
        learning_rate=CONFIG["LR"],
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type=CONFIG["LR_SCHED"],
        seed=42,
        output_dir=CONFIG["OUTPUT_DIR"],
        save_steps=CONFIG["SAVE_STEPS"],
        eval_strategy="steps",
        eval_steps=CONFIG["EVAL_STEPS"],
        save_total_limit=2,
        load_best_model_at_end=True,
        report_to="none",
    ),
)

print(f"🚀 НАЧАЛО ОБУЧЕНИЯ ({CONFIG['EPOCHS']} эпохи)...")
print(f"   Эффективный batch: {CONFIG['BATCH']*CONFIG['GRAD_ACC']}")
print(f"   Шагов на эпоху: {len(train_ds) // (CONFIG['BATCH']*CONFIG['GRAD_ACC'])}")

stats = trainer.train()
print(f"\n✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"   Loss: {stats.training_loss:.4f}")
print(f"   Шагов: {stats.global_step}")


In [ ]:
# ═══════════════════════════════════════════
# ЯЧЕЙКА 6: ТЕСТ КАЧЕСТВА
# ═══════════════════════════════════════════
FastLanguageModel.for_inference(model)

test_prompts = [
    "Включи свет на кухне",
    "Кто ты и что умеешь?",
    "Статус всех машин ARGOS",
    "Выключи весь свет",
    "Температура в квартире?",
]

print("🧪 ТЕСТ МОДЕЛИ:")
for prompt in test_prompts:
    inputs = tokenizer.apply_chat_template([
        {"role": "system", "content": CONFIG["SYSTEM_PROMPT"]},
        {"role": "user", "content": prompt},
    ], tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=128,
        temperature=0.3, do_sample=True,
        use_cache=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\nВопрос: {prompt}")
    print(f"Ответ: {response[:200]}")
    print("-" * 40)


In [ ]:
# ═══════════════════════════════════════════
# ЯЧЕЙКА 7: ЭКСПОРТ GGUF + ПУШ НА HF
# ═══════════════════════════════════════════
from huggingface_hub import login, HfApi
import os

# Логин с токеном из Secrets
login(token=CONFIG["HF_TOKEN"])
print("✅ HF login OK")

# 1. Сохраняем LoRA адаптер
adapter_path = f"{CONFIG['OUTPUT_DIR']}/adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"✅ Адаптер сохранён: {adapter_path}")

# 2. Пушим адаптер на HF
model.push_to_hub(CONFIG["HF_REPO"], token=CONFIG["HF_TOKEN"])
tokenizer.push_to_hub(CONFIG["HF_REPO"], token=CONFIG["HF_TOKEN"])
print(f"✅ Адаптер на HF: {CONFIG['HF_REPO']}")

# 3. GGUF экспорт через Unsloth (если доступен)
try:
    print("Экспортируем GGUF Q4_K_M...")
    model.save_pretrained_gguf(
        f"{CONFIG['OUTPUT_DIR']}/gguf",
        tokenizer,
        quantization_method="q4_k_m"
    )
    # Загружаем GGUF на HF
    api = HfApi()
    gguf_file = f"{CONFIG['OUTPUT_DIR']}/gguf/unsloth.Q4_K_M.gguf"
    if os.path.exists(gguf_file):
        api.upload_file(
            path_or_fileobj=gguf_file,
            path_in_repo="argos-v1-Q4_K_M.gguf",
            repo_id=CONFIG["HF_REPO"],
            token=CONFIG["HF_TOKEN"],
        )
        print(f"✅ GGUF Q4_K_M загружен!")
    
    # Q8_0 тоже
    model.save_pretrained_gguf(
        f"{CONFIG['OUTPUT_DIR']}/gguf8",
        tokenizer,
        quantization_method="q8_0"
    )
    gguf8 = f"{CONFIG['OUTPUT_DIR']}/gguf8/unsloth.Q8_0.gguf"
    if os.path.exists(gguf8):
        api.upload_file(
            path_or_fileobj=gguf8,
            path_in_repo="argos-v1-Q8_0-v2.gguf",
            repo_id=CONFIG["HF_REPO"],
            token=CONFIG["HF_TOKEN"],
        )
        print(f"✅ GGUF Q8_0 загружен!")
        
except Exception as e:
    print(f"⚠️ GGUF: {e}")
    print("Модель сохранена как LoRA адаптер на HF")

print(f"
🎉 Готово! https://huggingface.co/{CONFIG['HF_REPO']}")


In [ ]:
# ═══════════════════════════════════════════
# ЯЧЕЙКА 8: OLLAMA MODELFILE + ИНСТРУКЦИЯ
# ═══════════════════════════════════════════
modelfile = f"""FROM ./argos-v1-Q4_K_M.gguf

SYSTEM \"\"\"
{CONFIG["SYSTEM_PROMPT"]}
\"\"\"

PARAMETER temperature 0.3
PARAMETER num_ctx 2048
PARAMETER stop "<|im_end|>"
PARAMETER stop "</s>"
"""

with open(f"{CONFIG['OUTPUT_DIR']}/Modelfile", "w") as f:
    f.write(modelfile)

print("📋 Инструкция установки на ПК:")
print("="*50)
print(f"1. Скачать: https://huggingface.co/{CONFIG['HF_REPO']}/resolve/main/argos-v1-Q4_K_M.gguf")
print("2. Скачать Modelfile из репозитория")
print("3. ollama create argos-v1 -f Modelfile")
print("4. ollama run argos-v1 'Включи свет на кухне'")
print("="*50)
print()
print("📁 Файлы в /content/argos-v1-trained:")
import os
for f in os.listdir(CONFIG["OUTPUT_DIR"]):
    size = os.path.getsize(f"{CONFIG['OUTPUT_DIR']}/{f}")
    print(f"   {f}: {size/1e6:.1f} MB")

# Скачать через Colab
from google.colab import files
import zipfile, os

# Архивируем важные файлы
with zipfile.ZipFile("/content/argos_v1_result.zip", "w") as zf:
    for f in ["Modelfile", "adapter_config.json"]:
        fp = f"{CONFIG['OUTPUT_DIR']}/{f}"
        if os.path.exists(fp):
            zf.write(fp, f)

print("\n📥 Скачиваем результаты...")
files.download("/content/argos_v1_result.zip")
